# Step 4 — Vector Space Model dan Pembobotan TF-IDF

**Tujuan praktikum**
1. Mahasiswa memahami pembobotan *term frequency* (TF), *inverse document frequency* (IDF), dan TF-IDF.
2. Mahasiswa mampu merepresentasikan dokumen dan query sebagai vektor.
3. Mahasiswa mampu menghitung *cosine similarity* dan menghasilkan daftar dokumen terurut (*ranked retrieval*).

**Rumus yang digunakan**

$$w_{t,d} = (1 + \log_{10} tf_{t,d}) \times \log_{10}\frac{N}{df_t}$$

$$\cos(\vec q, \vec d) = \frac{\vec q \cdot \vec d}{\lVert \vec q \rVert \, \lVert \vec d \rVert}$$

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
import math
import numpy as np
from collections import Counter
from stki_utils import corpus_texts, preprocess, build_inverted_index, load_corpus

judul = dict(zip(load_corpus().doc_id, load_corpus().judul))
docs = {d: preprocess(t) for d, t in corpus_texts().items()}
index = build_inverted_index(docs)
N = len(docs)
vocab = list(index)
print(N, "dokumen,", len(vocab), "term")

30 dokumen, 472 term


## Langkah 1 — Term frequency dan variasinya

In [3]:
tf_d10 = Counter(docs["D10"])
pd.DataFrame({
    "tf mentah": tf_d10,
    "tf log (1+log10 tf)": {t: 1 + math.log10(v) for t, v in tf_d10.items()},
}).sort_values("tf mentah", ascending=False).head(8)

,tf mentah,tf log (1+log10 tf)
tani,3,1.477121
sensor,2,1.301030
cerdas,2,1.301030
data,1,1.000000
air,1,1.000000
guna,1,1.000000
efisiensi,1,1.000000
tingkat,1,1.000000


## Langkah 2 — Document frequency dan IDF
Term yang muncul di banyak dokumen kurang mampu membedakan dokumen sehingga diberi bobot IDF kecil.

In [6]:
idf = {t: math.log10(N / len(p)) for t, p in index.items()}
tabel_idf = pd.DataFrame({"df": {t: len(index[t]) for t in vocab}, "idf": idf})
pd.concat([tabel_idf.sort_values("df", ascending=False).head(5),
           tabel_idf.sort_values("df").head(5)]).round(3)

,df,idf
guna,11,0.436
lalu,9,0.523
bantu,8,0.574
tani,7,0.632
data,6,0.699
150,1,1.477
padu,1,1.477
padi,1,1.477
otot,1,1.477
otomatis,1,1.477


## Langkah 3 — Matriks TF-IDF dokumen

In [7]:
def tfidf_vector(tokens):
    return {t: (1 + math.log10(tf)) * idf[t] for t, tf in Counter(tokens).items() if t in idf}

M = pd.DataFrame(0.0, index=list(docs), columns=vocab)
for d, tokens in docs.items():
    for t, w in tfidf_vector(tokens).items():
        M.loc[d, t] = w
print("Ukuran matriks TF-IDF:", M.shape)
M.loc[["D07", "D08", "D09", "D10"], ["tani", "air", "pupuk", "sensor", "padi"]].round(3)

Ukuran matriks TF-IDF: (30, 472)


,tani,air,pupuk,sensor,padi
D07,0.632,0.875,0.000,0.00,2.182
D08,0.822,0.000,1.737,0.00,0.000
D09,0.632,0.875,0.000,0.00,0.000
D10,0.934,0.875,1.176,1.53,0.000


## Langkah 4 — Vektor query dan cosine similarity

In [9]:
def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return float(u @ v / (nu * nv)) if nu and nv else 0.0

query = "teknologi pertanian modern"
q_tokens = preprocess(query)
q_vec = pd.Series(0.0, index=vocab)
for t, w in tfidf_vector(q_tokens).items():
    q_vec[t] = w
print("Term query:", q_tokens, "| term di luar kosakata:", [t for t in q_tokens if t not in idf])

skor = {d: cosine(q_vec.values, M.loc[d].values) for d in docs}
hasil = sorted(skor.items(), key=lambda x: (-x[1], x[0]))[:5]
pd.DataFrame([(d, judul[d], round(s, 4)) for d, s in hasil], columns=["doc_id", "judul", "cosine"])

Term query: ['teknologi', 'tani', 'modern'] | term di luar kosakata: ['modern']


,doc_id,judul,cosine
0,D10,Pertanian Cerdas Berbasis IoT,0.2158
1,D06,Layanan Telemedisin,0.1442
2,D27,Literasi Digital untuk Pelajar,0.1421
3,D12,Harga Gabah Saat Panen Raya,0.0809
4,D08,Membuat Pupuk Organik,0.0670


## Langkah 5 — Implementasi efisien dengan inverted index
Menghitung cosine terhadap semua dokumen boros. Dengan inverted index, cukup dokumen yang memuat minimal satu term query yang diberi skor (*term-at-a-time*). Implementasi ini tersedia sebagai `TfidfModel` di `stki_utils.py`.

In [10]:
from stki_utils import TfidfModel

model = TfidfModel(docs)

def cari(query, k=5):
    hasil = model.search(preprocess(query), k=k)
    return pd.DataFrame([(d, judul[d], round(s, 4)) for d, s in hasil], columns=["doc_id", "judul", "skor"])

cek = model.search(q_tokens, k=5)
print("Sama dengan perhitungan matriks:", [d for d, _ in cek] == [d for d, _ in hasil])
cari("pemasaran digital UMKM")

Sama dengan perhitungan matriks: True


,doc_id,judul,skor
0,D19,Digitalisasi UMKM,0.3999
1,D22,Kredit Usaha Rakyat,0.1612
2,D24,Investasi Saham bagi Pemula,0.1103
3,D27,Literasi Digital untuk Pelajar,0.0819
4,D21,Pembayaran Digital dengan QRIS,0.0790


In [11]:
cari("pembelajaran daring mahasiswa")

,doc_id,judul,skor
0,D25,Pembelajaran Daring di Perguruan Tinggi,0.3535
1,D29,Perpustakaan Digital,0.1203
2,D28,Beasiswa Pendidikan Tinggi,0.1124
3,D26,Kurikulum Merdeka,0.0595
4,D14,Kecerdasan Buatan dalam Kehidupan,0.0587


## Tugas Step 4
1. Hitung secara manual (di kertas atau sel Markdown) bobot TF-IDF term `tani` pada dokumen D10, lalu cocokkan dengan hasil program.
2. Bandingkan tiga skema TF: mentah, logaritmik, dan biner. Untuk query `harga hasil panen petani`, bagaimana urutan 5 dokumen teratas berubah?
3. Query `pembelajaran daring mahasiswa` mengembalikan D14 (Kecerdasan Buatan). Jelaskan penyebabnya dari sisi preprocessing dan usulkan perbaikannya.
4. (Opsional) Bandingkan hasil Anda dengan `TfidfVectorizer` dari scikit-learn. Parameter apa yang harus diatur agar hasilnya mendekati implementasi manual?

In [9]:
# Tulis jawaban Anda di sini